# Inhibitory Two-Edge Triad Sensitivity

This notebook consolidates the previous self-free and self-inclusive two-edge triad notebooks into one sensitivity analysis. It keeps the induced connected two-edge triad census as the primary local motif analysis, then compares how appending nonzero self-connections changes edge participation, ablation, aggregation, and Schur-style reduction.


## tl;dr

The base connected two-edge triad set is defined without diagonal edges, so the Rees `A` / `B` / `C` motif census is the same in both variants. The `self_free` variant is the cleaner local topology analysis: every retained triad has exactly two directed non-self edges. The `with_self_connections` variant is best treated as a sensitivity analysis, because diagonal terms do not change motif labels but can substantially change weight rankings, edge participation, aggregate-matrix reductions, and ablation effects.


## Context & Methods

This analysis should run after the two global notebooks. It does not require their saved outputs, but it uses the same matrix orientation, E/I labeling assumptions, and helper functions. The notebook enumerates induced three-node subgraphs with exactly two non-self directed edges, maps those motifs onto Rees superpatterns, then reruns selected summaries after appending each triad node's nonzero self-connection.

### Key Assumptions

- The input matrix is configured as rows=senders/presynaptic and columns=receivers/postsynaptic, then transposed internally by `load_connectivity` so helper functions use rows=postsynaptic receivers and columns=presynaptic sources.
- Motif labels are assigned from the two non-self edges only. Self-connections are appended after triad selection and should be interpreted as a sensitivity layer rather than a different triad census.
- The null-model sections are exploratory by default because `N_NULL` is intentionally small enough for quick iteration.


## Setup

Import helper functions, define matrix paths, and choose the two self-connection variants to compare.


In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
for candidate in (PROJECT_ROOT, PROJECT_ROOT / "notebooks"):
    if str(candidate) not in sys.path:
        sys.path.append(str(candidate))

from inhibitory_modulation import (
    REES_SUPERPATTERN_NAMES,
    TRIAD_CENSUS_TO_REES_SUPERPATTERN,
    enumerate_two_edge_triads,
    load_connectivity,
    summarize_blocks,
    summarize_triad_categories,
    triad_ablation_analysis,
    triad_ablation_significance,
    triad_edge_participation,
    triad_enrichment_significance,
    triad_schur_decomposition,
)

pd.set_option("display.max_rows", 80)
pd.set_option("display.max_columns", 60)
pd.set_option("display.precision", 4)

CONNECTIVITY_PATH = PROJECT_ROOT / "matrices" / "mij_matrix.csv"
METADATA_NETLIST_PATH = PROJECT_ROOT / "matrices" / "mij_netlist.csv"
MATRIX_ORIENTATION = "pre_by_post"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "inhibitory_two_edge_triad_sensitivity"

CONNECTED_ONLY = True
N_NULL = 20
RANDOM_STATE = 0
SCHUR_REGULARIZATION = 1e-6
REGULARIZATION_GRID = np.array([1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1])

VARIANT_CONFIGS = {
    "self_free": False,
    "with_self_connections": True,
}

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CONNECTIVITY_PATH


## Data

Load the signed connectivity matrix and E/I labels. The triad enumeration below uses the same loaded matrix for both variants.


In [ ]:
data = load_connectivity(
    CONNECTIVITY_PATH,
    metadata_netlist_path=METADATA_NETLIST_PATH,
    matrix_orientation=MATRIX_ORIENTATION,
)

print(f"Loaded matrix: {data.source}")
print(f"Matrix shape: {data.matrix.shape[0]} x {data.matrix.shape[1]}")
display(data.classes.value_counts().rename(index={"e": "excitatory", "i": "inhibitory"}).to_frame("population_count"))
display(data.matrix.iloc[:5, :5])


## Results

### 1. Enumerate The Two Variants

The self-free variant is the primary motif census. The self-inclusive variant appends diagonal terms to those same selected triads.


In [ ]:
triads_by_variant = {}
overview_rows = []

for variant, include_self_connections in VARIANT_CONFIGS.items():
    triads = enumerate_two_edge_triads(
        data.matrix,
        data.classes,
        connected_only=CONNECTED_ONLY,
        include_self_connections=include_self_connections,
    )
    triads_by_variant[variant] = triads
    edge_count_distribution = triads["edge_count"].value_counts().sort_index().to_dict() if not triads.empty else {}
    overview_rows.append({
        "variant": variant,
        "include_self_connections": include_self_connections,
        "triad_count": len(triads),
        "unique_nonself_edges": int(triad_edge_participation(data.matrix, triads).query("edge_type == 'between_nodes'").shape[0]) if not triads.empty else 0,
        "unique_self_edges": int(triad_edge_participation(data.matrix, triads).query("edge_type == 'self'").shape[0]) if not triads.empty else 0,
        "edge_count_distribution": edge_count_distribution,
    })

overview = pd.DataFrame(overview_rows)
display(overview)


### 2. Compare Motif And E/I Composition

Motif labels should match across variants because they are based on non-self edges. Weight and local spectral summaries can differ once self-connections are appended.


In [ ]:
motif_summary = []
node_ei_summary = []
edge_ei_summary = []

for variant, triads in triads_by_variant.items():
    motif_table = summarize_triad_categories(
        triads,
        ["rees_superpattern", "motif", "rees_superpattern_name"],
    ).reset_index()
    motif_table.insert(0, "variant", variant)
    motif_summary.append(motif_table)

    node_table = summarize_triad_categories(triads, "node_ei_signature").reset_index()
    node_table.insert(0, "variant", variant)
    node_ei_summary.append(node_table)

    edge_table = summarize_triad_categories(triads, "edge_ei_signature").reset_index()
    edge_table.insert(0, "variant", variant)
    edge_ei_summary.append(edge_table)

motif_summary = pd.concat(motif_summary, ignore_index=True)
node_ei_summary = pd.concat(node_ei_summary, ignore_index=True)
edge_ei_summary = pd.concat(edge_ei_summary, ignore_index=True)

print("Rees motif summary")
display(motif_summary)
print("Node E/I composition")
display(node_ei_summary)
print("Top directed edge E/I signatures")
display(edge_ei_summary.groupby("variant", group_keys=False).head(12))


### 3. Region Signatures

Region signatures collapse each triad to the unique anatomical labels represented by its three populations.


In [ ]:
region_summary = []
within_region = []

for variant, triads in triads_by_variant.items():
    table = summarize_triad_categories(triads, "region_signature").reset_index()
    table.insert(0, "variant", variant)
    region_summary.append(table)

    rates = (
        triads.groupby("motif")
        .agg(
            triad_count=("triad_id", "count"),
            within_region_rate=("within_region", "mean"),
            median_region_count=("region_count", "median"),
        )
        .reset_index()
    )
    rates.insert(0, "variant", variant)
    within_region.append(rates)

region_summary = pd.concat(region_summary, ignore_index=True)
within_region = pd.concat(within_region, ignore_index=True)

print("Most frequent region signatures")
display(region_summary.groupby("variant", group_keys=False).head(15))
print("Within-region share by motif")
display(within_region.sort_values(["variant", "triad_count"], ascending=[True, False]))


### 4. Rank Triads And Participating Edges

This is where self-connections matter most: the same base motif can gain diagonal edges that affect total weight and edge participation.


In [ ]:
ranked_triads = []
edge_participation = []
ranking_columns = [
    "variant",
    "triad_id",
    "motif",
    "rees_superpattern",
    "rees_superpattern_name",
    "node_ei_signature",
    "edge_ei_signature",
    "region_signature",
    "node_a",
    "node_b",
    "node_c",
    "edge_count",
    "self_edge_count",
    "total_abs_weight",
]

for variant, triads in triads_by_variant.items():
    ranked = triads.sort_values("total_abs_weight", ascending=False).copy()
    ranked.insert(0, "variant", variant)
    ranked_triads.append(ranked[ranking_columns])

    participation = triad_edge_participation(data.matrix, triads)
    participation.insert(0, "variant", variant)
    edge_participation.append(participation)

ranked_triads = pd.concat(ranked_triads, ignore_index=True)
edge_participation = pd.concat(edge_participation, ignore_index=True)

print("Top triads by absolute weight")
display(ranked_triads.groupby("variant", group_keys=False).head(15))
print("Top participating directed edges")
display(edge_participation.groupby("variant", group_keys=False).head(15))


### 5. Enrichment Significance

Run a compact randomized-edge null for Rees motif labels and node E/I composition. Increase `N_NULL` before relying on p-values in a manuscript or presentation.


In [ ]:
enrichment_motif = []
enrichment_node_ei = []

for offset, (variant, include_self_connections) in enumerate(VARIANT_CONFIGS.items()):
    motif_table = triad_enrichment_significance(
        data.matrix,
        data.classes,
        by="rees_superpattern",
        n_null=N_NULL,
        random_state=RANDOM_STATE + offset,
        connected_only=CONNECTED_ONLY,
        include_self_connections=include_self_connections,
    )
    motif_table.insert(0, "variant", variant)
    motif_table["rees_superpattern_name"] = motif_table["rees_superpattern"].map(REES_SUPERPATTERN_NAMES)
    enrichment_motif.append(motif_table)

    node_table = triad_enrichment_significance(
        data.matrix,
        data.classes,
        by="node_ei_signature",
        n_null=N_NULL,
        random_state=RANDOM_STATE + 10 + offset,
        connected_only=CONNECTED_ONLY,
        include_self_connections=include_self_connections,
    )
    node_table.insert(0, "variant", variant)
    enrichment_node_ei.append(node_table)

enrichment_motif = pd.concat(enrichment_motif, ignore_index=True)
enrichment_node_ei = pd.concat(enrichment_node_ei, ignore_index=True)

print(f"Null draws per enrichment table: {N_NULL}")
display(enrichment_motif)
display(enrichment_node_ei)


### 6. Ablation Significance

Ablation removes directed edges that participate in the enumerated triads and recomputes whole-system stability diagnostics. Compare the self-inclusive and self-free versions to see how much diagonal terms drive the effect.


In [ ]:
global_ablation = []
motif_ablation = []
motif_ablation_significance = []

for offset, (variant, triads) in enumerate(triads_by_variant.items()):
    global_table = triad_ablation_analysis(data.matrix, triads).reset_index()
    global_table.insert(0, "variant", variant)
    global_ablation.append(global_table)

    top_superpatterns = (
        motif_summary.loc[motif_summary["variant"] == variant, "rees_superpattern"]
        .drop_duplicates()
        .tolist()
    )
    motif_table = triad_ablation_analysis(
        data.matrix,
        triads,
        category_col="rees_superpattern",
        categories=top_superpatterns,
    ).reset_index()
    motif_table.insert(0, "variant", variant)
    motif_ablation.append(motif_table)

    sig_table = triad_ablation_significance(
        data.matrix,
        triads,
        category_col="rees_superpattern",
        categories=top_superpatterns,
        n_null=N_NULL,
        random_state=RANDOM_STATE + 20 + offset,
    )
    sig_table.insert(0, "variant", variant)
    sig_table["rees_superpattern_name"] = sig_table["category"].map(REES_SUPERPATTERN_NAMES)
    motif_ablation_significance.append(sig_table)

global_ablation = pd.concat(global_ablation, ignore_index=True)
motif_ablation = pd.concat(motif_ablation, ignore_index=True)
motif_ablation_significance = pd.concat(motif_ablation_significance, ignore_index=True)

print("All triad-participating edge ablations")
display(global_ablation)
print("Rees-superpattern-specific edge ablations")
display(motif_ablation)
print(f"Random edge-set null draws per superpattern: {N_NULL}")
display(motif_ablation_significance)


### 7. Schur Reduction Sensitivity

Apply the E/I Schur-complement reduction to the triad aggregate and sweep the inhibitory-block regularization. Use this as a diagnostic ranking, especially when self-connections are included.


In [ ]:
schur_summary = []
schur_sensitivity = []

for variant, triads in triads_by_variant.items():
    triad_schur = triad_schur_decomposition(
        data.matrix,
        data.classes,
        triads,
        regularization=SCHUR_REGULARIZATION,
        normalize="triad_count",
    )
    block_table = summarize_blocks(triad_schur.blocks).reset_index()
    block_table.insert(0, "variant", variant)
    schur_summary.append(block_table)

    for regularization in REGULARIZATION_GRID:
        decomp = triad_schur_decomposition(
            data.matrix,
            data.classes,
            triads,
            regularization=float(regularization),
            normalize="triad_count",
        )
        row = decomp.effective_stability.copy()
        row["variant"] = variant
        row["regularization"] = regularization
        schur_sensitivity.append(row)

schur_summary = pd.concat(schur_summary, ignore_index=True)
schur_sensitivity = pd.DataFrame(schur_sensitivity)

print("Triad-aggregate E/I block summaries")
display(schur_summary)
print("Schur regularization sensitivity")
display(schur_sensitivity.set_index(["variant", "regularization"]))


### 8. Save Tables

Save the consolidated sensitivity outputs so the triad analysis has the same handoff pattern as the two global notebooks.


In [ ]:
overview.to_csv(OUTPUT_DIR / "triad_variant_overview.csv", index=False)
motif_summary.to_csv(OUTPUT_DIR / "triad_motif_summary_by_variant.csv", index=False)
node_ei_summary.to_csv(OUTPUT_DIR / "triad_node_ei_summary_by_variant.csv", index=False)
edge_ei_summary.to_csv(OUTPUT_DIR / "triad_edge_ei_summary_by_variant.csv", index=False)
region_summary.to_csv(OUTPUT_DIR / "triad_region_summary_by_variant.csv", index=False)
within_region.to_csv(OUTPUT_DIR / "triad_within_region_by_motif.csv", index=False)
ranked_triads.to_csv(OUTPUT_DIR / "top_triads_by_variant.csv", index=False)
edge_participation.to_csv(OUTPUT_DIR / "triad_edge_participation_by_variant.csv", index=False)
enrichment_motif.to_csv(OUTPUT_DIR / "triad_motif_enrichment_by_variant.csv", index=False)
enrichment_node_ei.to_csv(OUTPUT_DIR / "triad_node_ei_enrichment_by_variant.csv", index=False)
global_ablation.to_csv(OUTPUT_DIR / "triad_global_ablation_by_variant.csv", index=False)
motif_ablation.to_csv(OUTPUT_DIR / "triad_motif_ablation_by_variant.csv", index=False)
motif_ablation_significance.to_csv(OUTPUT_DIR / "triad_motif_ablation_significance_by_variant.csv", index=False)
schur_summary.to_csv(OUTPUT_DIR / "triad_schur_block_summary_by_variant.csv", index=False)
schur_sensitivity.to_csv(OUTPUT_DIR / "triad_schur_regularization_sensitivity_by_variant.csv", index=False)

for variant, triads in triads_by_variant.items():
    triads.to_csv(OUTPUT_DIR / f"enumerated_triads_{variant}.csv", index=False)

print(f"Saved consolidated triad sensitivity tables to {OUTPUT_DIR.resolve()}")


## Takeaways

- Use `self_free` as the primary two-edge motif census because it preserves the exact local topology definition.
- Use `with_self_connections` as a sensitivity layer for weight rankings, edge participation, ablation, aggregation, and Schur reduction.
- Do not interpret unchanged Rees counts as evidence that self-connections are irrelevant; they are unchanged because the labels are defined only by the two non-self directed edges.
- Keep this notebook downstream of the global fracture analyses: it explains local motif composition but does not need to run before the EE/EI/IE/II or backbone ablation notebooks.


## References

- Holland, P. W., & Leinhardt, S. (1974). *The Statistical Analysis of Local Structure in Social Networks*. NBER Working Paper 0044. https://doi.org/10.3386/w0044
- NetworkX documentation: `triadic_census`, including the 16 directed triad-census labels used here. https://networkx.org/documentation/stable/reference/algorithms/generated/networkx.algorithms.triads.triadic_census.html
- Rees, C. L., Wheeler, D. W., Hamilton, D. J., White, C. M., Komendantov, A. O., & Ascoli, G. A. (2016). *Graph Theoretic and Motif Analyses of the Hippocampal Neuron Type Potential Connectome*. eNeuro, 3(6), ENEURO.0205-16.2016. https://www.eneuro.org/content/3/6/ENEURO.0205-16.2016
- Zhang, F. (Ed.). (2005). *The Schur Complement and Its Applications*. Springer. https://doi.org/10.1007/b105056
- Golub, G. H., & Van Loan, C. F. (2013). *Matrix Computations* (4th ed.). Johns Hopkins University Press. https://www.press.jhu.edu/books/title/10678/matrix-computations
